In [0]:
%sql
-- 1. Crear el catálogo del equipo
CREATE CATALOG IF NOT EXISTS stasys_team2;

-- 2. Seleccionar el catálogo
USE CATALOG stasys_team2;

-- ----------------------------------------------------
-- ESQUEMA DE TEAM 02: 
-- ----------------------------------------------------

-- 3. Crear tu esquema de trabajo
CREATE SCHEMA IF NOT EXISTS gianmarco;

-- 4. Verificar los esquemas disponibles
SHOW SCHEMAS IN stasys_team2;

-- 5. Comprobar permisos y ubicación
SELECT current_catalog(), current_schema();

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
    stasys_team2.gianmarco.consultas_denuncias_silver
USING CSV
OPTIONS (
    header = "true",
    inferSchema = "true"
)
LOCATION 's3://data-lagoon-y/silver/cleaned/consultas_denuncias_2026_v1/2026/10/13/20261013_150857_891495/CONSULTAS_DENUNCIAS_2026_v1_silver.csv';

In [0]:
%sql
-- Confirmar que la tabla existe.
DESCRIBE TABLE
    stasys_team2.gianmarco.consultas_denuncias_silver;

-- Consultar una muestra.
SELECT *
FROM stasys_team2.gianmarco.consultas_denuncias_silver
LIMIT 20;

-- Contar los registros.
SELECT COUNT(*) AS total_registros
FROM stasys_team2.gianmarco.consultas_denuncias_silver;


In [0]:
%sql
-- Consulta 1 — Cantidad de solicitudes por tipo

SELECT
    DE_TIPO_SOLICITUD AS tipo_solicitud,
    COUNT(*) AS total_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
GROUP BY DE_TIPO_SOLICITUD
ORDER BY total_solicitudes DESC;

In [0]:
%sql
-- Consulta 2 — Evolución mensual de las solicitudes

SELECT
    DATE_TRUNC('MONTH', FE_REGISTRO_FECHA) AS mes,
    COUNT(*) AS cantidad_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
WHERE FE_REGISTRO_FECHA IS NOT NULL
GROUP BY DATE_TRUNC('MONTH', FE_REGISTRO_FECHA)
ORDER BY cantidad_solicitudes DESC;

In [0]:
%sql
-- Consulta 3 — Ranking de las diez instituciones con más solicitudes

SELECT
    DE_INSTITUCION,
    DE_TIPO_INST,
    COUNT(*) AS cantidad_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
WHERE DE_INSTITUCION IS NOT NULL
GROUP BY
    DE_INSTITUCION,
    DE_TIPO_INST
ORDER BY cantidad_solicitudes DESC
LIMIT 10;


In [0]:
%sql
-- Consulta 4 — Comparación de solicitudes según la naturaleza de la institución


SELECT
    DE_NATURALEZA_INST,
    COUNT(*) AS total_solicitudes,
    COUNT(DISTINCT DE_INSTITUCION)
        AS instituciones_distintas,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS porcentaje_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
WHERE DE_NATURALEZA_INST IS NOT NULL
GROUP BY DE_NATURALEZA_INST
ORDER BY total_solicitudes DESC;


In [0]:
%sql
SELECT 
  CASE 
    WHEN `DE_SOLICITANTE_EDAD` < 0 OR `DE_SOLICITANTE_EDAD` > 120 THEN 'Edad no válida'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 0 AND 17 THEN '0-17 años (Menor)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 18 AND 29 THEN '18-29 años (Joven)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 30 AND 44 THEN '30-44 años (Adulto joven)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 45 AND 59 THEN '45-59 años (Adulto)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 60 AND 74 THEN '60-74 años (Adulto mayor)'
    WHEN `DE_SOLICITANTE_EDAD` >= 75 THEN '75+ años (Anciano)'
    ELSE 'Sin clasificar'
  END AS grupo_edad,
  `DE_SOLICITANTE_GENERO` AS genero,
  COUNT(*) AS total_solicitudes,
  ROUND(AVG(`DE_SOLICITANTE_EDAD`), 1) AS edad_promedio,
  MIN(`DE_SOLICITANTE_EDAD`) AS edad_minima,
  MAX(`DE_SOLICITANTE_EDAD`) AS edad_maxima,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje_total
FROM `stasys_team2`.`gianmarco`.`consultas_denuncias_silver`
WHERE `DE_SOLICITANTE_GENERO` IS NOT NULL 
  AND `DE_SOLICITANTE_EDAD` IS NOT NULL
GROUP BY 
  CASE 
    WHEN `DE_SOLICITANTE_EDAD` < 0 OR `DE_SOLICITANTE_EDAD` > 120 THEN 'Edad no válida'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 0 AND 17 THEN '0-17 años (Menor)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 18 AND 29 THEN '18-29 años (Joven)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 30 AND 44 THEN '30-44 años (Adulto joven)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 45 AND 59 THEN '45-59 años (Adulto)'
    WHEN `DE_SOLICITANTE_EDAD` BETWEEN 60 AND 74 THEN '60-74 años (Adulto mayor)'
    WHEN `DE_SOLICITANTE_EDAD` >= 75 THEN '75+ años (Anciano)'
    ELSE 'Sin clasificar'
  END,
  `DE_SOLICITANTE_GENERO`
ORDER BY 
  CASE 
    WHEN grupo_edad = '0-17 años (Menor)' THEN 1
    WHEN grupo_edad = '18-29 años (Joven)' THEN 2
    WHEN grupo_edad = '30-44 años (Adulto joven)' THEN 3
    WHEN grupo_edad = '45-59 años (Adulto)' THEN 4
    WHEN grupo_edad = '60-74 años (Adulto mayor)' THEN 5
    WHEN grupo_edad = '75+ años (Anciano)' THEN 6
    WHEN grupo_edad = 'Edad no válida' THEN 7
    ELSE 8
  END,
  genero